# EIS Scenario Template

# Establish variables and environment

In [ ]:
# import packages
import pandas as pd
import pathlib
from pathlib import Path
import os
import sys
import arcpy
import numpy as np
import pickle
# external connection packages
from sqlalchemy.engine import URL
from sqlalchemy import create_engine
# add scripts folder to path so utils and forecast_functions can be imported
sys.path.insert(0, str(pathlib.Path().absolute().parent / 'scripts'))
from utils import *
from forecast_functions import *
# pandas options
pd.options.mode.copy_on_write = True
pd.options.mode.chained_assignment = None
pd.options.display.max_columns = 999
pd.options.display.max_rows    = 999

# my workspace 
workspace = r"C:\GIS\Scratch.gdb"
# current working directory
local_path = pathlib.Path().absolute()

# get bonus_condit
# set data path as a subfolder of the current working directory 
data_dir = local_path.parents[0] / 'data'
# folder to save processed data
out_dir  = local_path.parents[0] / 'data/processed_data'

base_data_dir = local_path.parents[0].joinpath('data/processed_data/base_data')
# workspace gdb for stuff that doesnt work in memory
# gdb = os.path.join(local_path,'Workspace.gdb')
gdb = workspace
# set environement workspace to in memory 
arcpy.env.workspace = 'memory'
# # clear memory workspace
# arcpy.management.Delete('memory')

# overwrite true
arcpy.env.overwriteOutput = True
# Set spatial reference to NAD 1983 UTM Zone 10N
sr = arcpy.SpatialReference(26910)

# get parcels from the database
# network path to connection files
filePath = "F:/GIS/PARCELUPDATE/Workspace/"
# database file path 
sdeBase    = os.path.join(filePath, "Vector.sde")
sdeCollect = os.path.join(filePath, "Collection.sde")
sdeTabular = os.path.join(filePath, "Tabular.sde")
sdeEdit    = os.path.join(filePath, "Edit.sde")

# Pickle variables
# part 1 - spatial joins and new categorical fields
parcel_pickle_part1    = data_dir / 'parcel_pickle1.pkl'
# part 2 - forecasting applied
parcel_pickle_part2    = data_dir / 'parcel_pickle2.pkl'

# columsn to list
initial_columns = [ 'APN',
                    'APO_ADDRESS',
                    'Residential_Units',
                    'TouristAccommodation_Units',
                    'CommercialFloorArea_SqFt',
                    'YEAR',
                    'JURISDICTION',
                    'COUNTY',
                    'OWNERSHIP_TYPE',
                    'COUNTY_LANDUSE_DESCRIPTION',
                    'EXISTING_LANDUSE',
                    'REGIONAL_LANDUSE',
                    'YEAR_BUILT',
                    'PLAN_ID',
                    'PLAN_NAME',
                    'ZONING_ID',
                    'ZONING_DESCRIPTION',
                    'TOWN_CENTER',
                    'LOCATION_TO_TOWNCENTER',
                    'TAZ',
                    'PARCEL_ACRES',
                    'PARCEL_SQFT',
                    'WITHIN_BONUSUNIT_BNDY',
                    'WITHIN_TRPA_BNDY']


## Establish Scenario Conditions

In [ ]:
scenario_name = 'Alternative_1'
folder_2035 = out_dir / (scenario_name + '_2035')
folder_2050 = out_dir / (scenario_name + '_2050')
# create folders if they don't exist
folder_2035.mkdir(parents=True, exist_ok=True)
folder_2050.mkdir(parents=True, exist_ok=True)    

### Get base year parcel pickle and scenario residential zone development targets

In [ ]:
# Load base parcel data exported by parcel_engineering
#sdfParcels = pd.read_pickle(out_dir / 'base_parcel_data.pkl')
# temp hardcoded workaround
parcel_data_path = base_data_dir / 'base_parcel_data.parquet'
sdfParcels = pd.read_parquet(parcel_data_path)
# Load lookup lists

res_zoned_lookup      = "inputs/forecast_residential_zoned_units.csv"

dfResZoned    = pd.read_csv(res_zoned_lookup)

### Establish scenario proportions

In [ ]:
# --- Pool type proportions by (Jurisdiction, Unit_Pool) ---
# Any combination not listed falls back to 'default'
zone_proportions = {
    # ('CSLT', 'Bonus Unit'): {'mf': 0.50, 'sf': 0.35, 'infill': 0.15},
    # ('TRPA', 'General'):    {'mf': 0.40, 'sf': 0.45, 'infill': 0.15},
    'default': {'mf': 0.35, 'sf': 0.50, 'infill': 0.15},
}

# --- Occupancy rates by FORECAST_REASON keyword ---
occupancy_rates = {
    'Bonus':   1.00,
    'CTC':     1.00,
    'General': 0.35,
    'ADU': .70
}

# --- Household income proportions by FORECAST_REASON keyword ---
income_proportions = {
    'Bonus':   {'low': 0.78, 'medium': 0.20, 'high': 0.02},
    'General': {'low': 0.01, 'medium': 0.02, 'high': 0.97},
    'ADU':     {'low': 0.65, 'medium': 0.20, 'high': 0.15},
    'CTC':     {'low': 1.00, 'medium': 0.00, 'high': 0.00},
}

# ── Pool computation ──────────────────────────────────────────────────────────
dfPool = pd.read_csv(res_zoned_lookup)
dfPool = get_adjusted_future_units(dfPool, zone_proportions)

In [ ]:
conditions = get_parcel_conditions()
sdfParcels, df_built_parcels = forecast_jurisdiction_pools(sdfParcels, dfPool, conditions)
sdfParcels, df_built_parcels = forecast_trpa_pools(sdfParcels, dfPool, conditions, df_built_parcels)
sdfParcels, df_built_parcels =assign_remainders(sdfParcels, conditions, df_built_parcels, adu_target=4385)
df_res_assigned_lookup = data_dir / "inputs/res_assigned_lookup.csv"
df_forecast_check = check_forecast_results(sdfParcels, dfPool)
sdfParcels = assign_development_year(sdfParcels, dfResZoned)

# gotta sort this one out. Need  to make lookup list
sdfParcels, dfResAssigned = assign_occupancy_rate(sdfParcels, occupancy_rates, df_res_assigned_lookup)
sdfParcels = assign_income_categories(sdfParcels, dfResAssigned, income_proportions)

In [ ]:
# This should be the new base year SocioEon_Summer
socio            = r'Base\data\processed_data\SocioEcon_Summer.csv'
socio_path       = local_path.parents[1].joinpath(socio)
dfSocio          = pd.read_csv(socio_path)
# rename taz column to be capitalized for merging
dfSocio.rename(columns={'taz': 'TAZ'}, inplace=True)
# change taz column to object for merging
dfSocio['TAZ'] = dfSocio['TAZ'].astype(str)

# This needs to increase proportionally to population
school = r'Base\data\processed_data\SchoolEnrollment.csv'
school_path = local_path.parents[1].joinpath(school)
dfSchool = pd.read_csv(school_path)



In [ ]:
dfTAZ_Summary_2035, dfTAZ_Summary_2050 = build_taz_summary(sdfParcels,dfSocio)
# add date to filename

date_str = datetime.now().strftime("%Y-%m-%d")
sdfParcels.to_parquet(data_folder / f'sdfParcels_{date_str}.parquet', index=False)
dfTAZ_Summary_2035.to_csv(rf'data/TAZ_Summary_2035_{date_str}.csv', index=False)
dfTAZ_Summary_2050.to_csv(rf'data/TAZ_Summary_2050_{date_str}.csv', index=False)

In [ ]:
taz_field_mapping = {
 'TAZ': 'TAZ',
 'TOTAL_FORECASTED_RESIDENTIAL_UNITS': 'total_residential_units',
 'NEW_OCCUPANCY_RATE': 'census_occ_rate',
 'TOTAL_FORECASTED_UNITS_OCCUPIED': 'total_occ_units',
 'TOTAL_FORECASTED_UNITS_LOW_INCOME': 'occ_units_low_inc',
 'TOTAL_FORECASTED_UNITS_MED_INCOME': 'occ_units_med_inc',
 'TOTAL_FORECASTED_UNITS_HIGH_INCOME': 'occ_units_high_inc',
 'persons_per_occ_unit': 'persons_per_occ_unit',
 'TOTAL_FORECASTED_PERSONS': 'total_persons',
 'emp_retail': 'emp_retail',
 'emp_srvc': 'emp_srvc',
 'emp_rec': 'emp_rec',
 'emp_game': 'emp_game',
 'emp_other': 'emp_other'
}
df_taz_2035 = clean_taz_summary(dfTAZ_Summary_2035, taz_field_mapping)
df_taz_2050 = clean_taz_summary(dfTAZ_Summary_2050, taz_field_mapping)
target_sum_2035 = 55592
target_sum_2050 =57611
df_taz_2035 = adjust_taz_population(df_taz_2035, target_population=target_sum_2035)
df_taz_2050 = adjust_taz_population(df_taz_2050, target_population=target_sum_2050)



### Need to add code to update school enrollment and to populate employment 

In [ ]:
dfSchool_2035, dfSchool_2050 = adjust_school_enrollment(df_taz_2035, df_taz_2050, dfSocio, dfSchool)

### Just for this scenario export the adjusted persons per households to a lookup csv for consistency across models

In [ ]:
persons_per_occ_unit_2035 = df_taz_2035[["TAZ", "persons_per_occ_unit"]]
persons_per_occ_unit_2050 = df_taz_2050[["TAZ", "persons_per_occ_unit"]]
persons_per_occ_unit_2035_path = r'data\processed_data\base_data\persons_per_occ_unit_2035.csv'
persons_per_occ_unit_2050_path = r'data\processed_data\base_data\persons_per_occ_unit_2050.csv'
persons_per_occ_unit_2035_path = local_path.parents[0].joinpath(persons_per_occ_unit_2035_path)
persons_per_occ_unit_2035.to_csv(persons_per_occ_unit_2035_path, index=False)
persons_per_occ_unit_2050_path = local_path.parents[0].joinpath(persons_per_occ_unit_2050_path)
persons_per_occ_unit_2050.to_csv(persons_per_occ_unit_2050_path, index=False)

## Update employment numbers to match RTP

In [ ]:
employment_2035 = r'data\inputs\employment_2035.csv'
employment_2035_path = local_path.parents[0].joinpath(employment_2035)
dfEmployment_2035 = pd.read_csv(employment_2035_path)
employment_2050 = r'data\inputs\employment_2050.csv'
employment_2050_path = local_path.parents[0].joinpath(employment_2050)
dfEmployment_2050 = pd.read_csv(employment_2050_path)

df_taz_2035 = update_employment_numbers(df_taz_2035, dfEmployment_2035)
df_taz_2050 = update_employment_numbers(df_taz_2050, dfEmployment_2050)

In [ ]:
# rename TAZ to taz
df_taz_2035.rename(columns={'TAZ':'taz'}, inplace=True)
df_taz_2050.rename(columns={'TAZ':'taz'}, inplace=True)

In [ ]:
# This won't change
overnight_visitors = r'data\inputs\OvernightVisitorZonalData_Summer.csv'
overnight_visitors_path = local_path.parents[0].joinpath(overnight_visitors)
dfOvernightVisitors = pd.read_csv(overnight_visitors_path)

visitor_occupancy = r'Base\data\processed_data\VisitorOccupancyRates_Summer.csv'
visitor_occupancy_path = local_path.parents[1].joinpath(visitor_occupancy)
dfVisitorOccupancy = pd.read_csv(visitor_occupancy_path)

# export TAZ summaries to folders
df_taz_2035.to_csv(folder_2035 / 'SocioEcon_Summer.csv', index=False)
dfSchool_2035.to_csv(folder_2035 / 'SchoolEnrollment.csv', index=False)


dfSchool_2050.to_csv(folder_2050 / 'SchoolEnrollment.csv', index=False)
df_taz_2050.to_csv(folder_2050 / 'SocioEcon_Summer.csv', index=False)

dfOvernightVisitors.to_csv(folder_2035 / 'OvernightVisitorZonalData_Summer.csv', index=False)
dfOvernightVisitors.to_csv(folder_2050 / 'OvernightVisitorZonalData_Summer.csv', index=False)
dfVisitorOccupancy.to_csv(folder_2035 / 'VisitorOccupancyRates_Summer.csv', index=False)
dfVisitorOccupancy.to_csv(folder_2050 / 'VisitorOccupancyRates_Summer.csv', index=False)